# Movie Success Prediction — Model Training Notebook

Aligned with the **cahier des charges** (*Movie Success Prediction*, v1.0).

## What this notebook does
1. Loads the real TMDB-derived dataset (`movies_dataset.csv`).
2. Defines the success target via ROI: `success = 1 if ROI > 0` (profitable).
3. Engineers pre-release-only features (no revenue leakage at prediction time).
4. Trains and compares **Logistic Regression**, **Random Forest**, and **Gradient Boosting**.
5. Evaluates with Accuracy, Precision, Recall, F1, ROC-AUC vs a majority-class reference.
6. Saves the best model + preprocessor to `artifacts/`.
7. Runs sample inferences.


## 1. Setup and imports

Required: `pandas`, `numpy`, `scikit-learn`, `joblib`, `matplotlib`, `seaborn`.

```bash
pip install pandas numpy scikit-learn joblib matplotlib seaborn
```

In [ ]:
import pandas as pd
import numpy as np
import json
import math
import ast
import joblib
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style("whitegrid")

RANDOM_SEED = 42
TEST_SIZE = 0.2
SUCCESS_THRESHOLD = 0.0  # ROI > 0 => profitable => success
CSV_PATH = "../data/movies_dataset.csv"  # adjust if running elsewhere
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load data

The dataset is the TMDB-derived `movies_dataset.csv` with columns:
`title, overview, budget, revenue, runtime, genres, vote_average, vote_count, popularity, release_date, origin_country, original_language, production_companies, adult, director, top_cast`.

In [ ]:
def parse_list(s):
    try:
        v = ast.literal_eval(s)
        return [str(x) for x in v] if isinstance(v, list) else []
    except Exception:
        return []

df = pd.read_csv(CSV_PATH)
print("Raw shape:", df.shape)
print(df.head())

# Keep rows with valid pre-release features + revenue for target construction
df = df[
    (df["budget"] > 0)
    & (df["revenue"] > 0)
    & (df["runtime"] > 0)
    & df["release_date"].astype(str).str[:4].str.isdigit()
].copy()
print("Valid rows:", len(df))

# Target: ROI = (Revenue - Budget) / Budget ; success = 1 if ROI > 0
df["roi"] = (df["revenue"] - df["budget"]) / df["budget"]
df["success"] = (df["roi"] > SUCCESS_THRESHOLD).astype(int)

pos = int(df["success"].sum())
print(f"Success rate (ROI > {SUCCESS_THRESHOLD}): {pos}/{len(df)} = {pos/len(df)*100:.1f}%")
print("ROI min/median/max:", df["roi"].min().round(3), df["roi"].median().round(3), df["roi"].max().round(3))

## 3. Feature engineering

Features available **before release** (no leakage):
- `budget` → log1p-transformed then standard scaled
- `runtime`, `release_year`, `cast_count` → standard scaled
- `has_director` → binary
- `genre` → multi-hot (top genres only)
- `language`, `country` → one-hot (top categories + "Other")

Revenues are used **only** to build the target during training.

In [ ]:
df["release_year"] = df["release_date"].astype(str).str[:4].astype(int)
df["log_budget"] = np.log1p(df["budget"])
df["genres"] = df["genres"].apply(parse_list)
df["language"] = df["original_language"].fillna("en")
df["country"] = df["origin_country"].apply(lambda s: parse_list(s)[0] if parse_list(s) else "Unknown")
df["cast_count"] = df["top_cast"].apply(lambda s: len(parse_list(s)))
df["has_director"] = df["director"].notna() & (df["director"].astype(str).str.strip() != "")
df["has_director"] = df["has_director"].astype(int)

# Vocabularies from TRAIN only (to avoid leakage)
train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df["success"]
)

genre_counter = Counter(g for gs in train_df["genres"] for g in gs)
GENRE_VOCAB = [g for g, c in genre_counter.most_common() if c >= 30]

lang_counter = Counter(train_df["language"])
LANG_VOCAB = [l for l, _ in lang_counter.most_common(6)] + ["Other"]

country_counter = Counter(train_df["country"])
COUNTRY_VOCAB = [c for c, _ in country_counter.most_common(6)] + ["Other"]

print("Genre vocab:", GENRE_VOCAB)
print("Lang vocab:", LANG_VOCAB)
print("Country vocab:", COUNTRY_VOCAB)
print(f"Train {len(train_df)} / Test {len(test_df)}")

In [ ]:
NUMERIC_COLS = ["log_budget", "runtime", "release_year", "cast_count"]

numeric_stats = {}
for col in NUMERIC_COLS:
    vals = train_df[col].values
    numeric_stats[col] = {"mean": float(vals.mean()), "std": float(vals.std() if vals.std() > 0 else 1.0)}

def build_feature_vector(r):
    x = []
    for col in NUMERIC_COLS:
        st = numeric_stats[col]
        x.append((r[col] - st["mean"]) / st["std"])
    x.append(float(r["has_director"]))
    gset = set(r["genres"])
    x += [1.0 if g in gset else 0.0 for g in GENRE_VOCAB]
    lang = r["language"] if r["language"] in LANG_VOCAB[:-1] else "Other"
    x += [1.0 if lang == l else 0.0 for l in LANG_VOCAB]
    ctry = r["country"] if r["country"] in COUNTRY_VOCAB[:-1] else "Other"
    x += [1.0 if ctry == c else 0.0 for c in COUNTRY_VOCAB]
    return np.array(x, dtype=float)

feature_names = NUMERIC_COLS + ["has_director"] \
    + [f"genre__{g}" for g in GENRE_VOCAB] \
    + [f"lang__{l}" for l in LANG_VOCAB] \
    + [f"country__{c}" for c in COUNTRY_VOCAB]

X_train = np.vstack([build_feature_vector(r) for _, r in train_df.iterrows()])
y_train = train_df["success"].values
X_test = np.vstack([build_feature_vector(r) for _, r in test_df.iterrows()])
y_test = test_df["success"].values

print("Feature dim:", X_train.shape[1])
print("Train success rate:", round(y_train.mean(), 4))
print("Test success rate:", round(y_test.mean(), 4))

## 4. Train and compare models

Per the cahier des charges, compare Logistic Regression, Random Forest, and Gradient Boosting on the same split. Select the best by ROC-AUC.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def train_logistic(X, y, lr=0.15, iters=3000, l2=0.0002, seed=42):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(iters):
        z = X @ w + b
        p = sigmoid(z)
        err = p - y
        dw = (X.T @ err) / n + l2 * w
        db = err.mean()
        w -= lr * dw
        b -= lr * db
    return w, b

def eval_numpy(X, y, w, b, threshold=0.5):
    p = sigmoid(X @ w + b)
    pred = (p >= threshold).astype(int)
    acc = (pred == y).mean()
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    try:
        auc = roc_auc_score(y, p)
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "proba": p}

w, b = train_logistic(X_train, y_train)
res_train = eval_numpy(X_train, y_train, w, b)
res_test = eval_numpy(X_test, y_test, w, b)

print("Logistic Regression — Train:", {k: round(v, 4) for k, v in res_train.items() if k != "proba"})
print("Logistic Regression — Test :", {k: round(v, 4) for k, v in res_test.items() if k != "proba"})

In [ ]:
# Compare with scikit-learn pipelines
pipelines = {
    "Logistic Regression": Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
                ("scaler", StandardScaler()),
            ]), ["budget", "runtime", "release_year", "cast_count"]),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), ["genre", "language", "country"]),
        ])),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
                ("scaler", StandardScaler()),
            ]), ["budget", "runtime", "release_year", "cast_count"]),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), ["genre", "language", "country"]),
        ])),
        ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
                ("scaler", StandardScaler()),
            ]), ["budget", "runtime", "release_year", "cast_count"]),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), ["genre", "language", "country"]),
        ])),
        ("clf", GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_SEED)),
    ]),
}

results = {}
for name, pipe in pipelines.items():
    pipe.fit(train_df[NUMERIC_COLS + ["genre", "language", "country", "has_director"]], y_train)
    y_pred = pipe.predict(test_df[NUMERIC_COLS + ["genre", "language", "country", "has_director"]])
    y_proba = pipe.predict_proba(test_df[NUMERIC_COLS + ["genre", "language", "country", "has_director"]])[:, 1]
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    print(f"{name:22s} acc={results[name]['accuracy']:.3f} prec={results[name]['precision']:.3f} rec={results[name]['recall']:.3f} f1={results[name]['f1']:.3f} auc={results[name]['roc_auc']:.3f}")

best_name = max(results, key=lambda k: results[k]["roc_auc"])
print("\nBest model:", best_name, "| ROC-AUC:", round(results[best_name]["roc_auc"], 4))

## 5. Evaluation vs reference

Compare against a majority-class reference model (per the cahier des charges).

In [ ]:
majority = int(round(y_test.mean()))
y_ref = np.full_like(y_test, majority)
ref = {
    "accuracy": accuracy_score(y_test, y_ref),
    "precision": precision_score(y_test, y_ref, zero_division=0),
    "recall": recall_score(y_test, y_ref, zero_division=0),
    "f1": f1_score(y_test, y_ref, zero_division=0),
    "roc_auc": float("nan"),
}
print("Reference (majority class):", {k: (round(v, 4) if v == v else "N/A") for k, v in ref.items()})

# Bar chart of test ROC-AUC by model
plt.figure(figsize=(7, 4))
names = list(results.keys()) + ["Reference"]
aucs = [results[n]["roc_auc"] for n in results.keys()] + [ref["roc_auc"]]
sns.barplot(x=names, y=aucs, palette="viridis")
plt.ylim(0, 1)
plt.ylabel("ROC-AUC")
plt.title("Test ROC-AUC by model")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Save artifacts

Saves `model.joblib`, `preprocessor.joblib`, and `metadata.json` to `artifacts/`.
These are consumed by `ml-service/main.py` (FastAPI `POST /predict`).

In [ ]:
best_pipe = pipelines[best_name]

# Build a standalone preprocessor (same transform as the pipeline)
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]), ["budget", "runtime", "release_year", "cast_count"]),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), ["genre", "language", "country"]),
])
preprocessor.fit(train_df[NUMERIC_COLS + ["genre", "language", "country"]])

joblib.dump(best_pipe, ARTIFACTS_DIR / "model.joblib")
joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor.joblib")

metadata = {
    "version": "v1.0.0",
    "algorithm": best_name,
    "threshold": 0.5,
    "metrics": results[best_name],
    "reference": ref,
    "features": {
        "numeric": NUMERIC_COLS,
        "categorical": ["genre", "language", "country"],
        "binary": ["has_director"],
    },
    "target": "success",
    "success_rule": "ROI > 0 (revenue > budget => profitable)",
    "random_seed": RANDOM_SEED,
    "n_samples": int(len(df)),
    "train_size": int(len(train_df)),
    "test_size": int(len(test_df)),
}
with open(ARTIFACTS_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Saved {ARTIFACTS_DIR / 'model.joblib'}")
print(f"Saved {ARTIFACTS_DIR / 'preprocessor.joblib'}")
print(f"Saved {ARTIFACTS_DIR / 'metadata.json'}")

## 7. Inference example

Load the saved artifacts and predict on sample movies.

In [ ]:
import joblib
import numpy as np
import pandas as pd

pipe = joblib.load(ARTIFACTS_DIR / "model.joblib")

samples = pd.DataFrame([
    {"title": "Big Budget Action", "budget": 150_000_000, "runtime": 125, "release_year": 2024,
     "genre": "Action", "language": "en", "country": "US", "has_director": 1},
    {"title": "Low Budget Documentary", "budget": 2_000_000, "runtime": 95, "release_year": 2023,
     "genre": "Documentary", "language": "fr", "country": "FR", "has_director": 1},
    {"title": "Mid Budget Comedy", "budget": 30_000_000, "runtime": 100, "release_year": 2022,
     "genre": "Comedy", "language": "en", "country": "US", "has_director": 1},
])

feat_cols = NUMERIC_COLS + ["genre", "language", "country", "has_director"]
probas = pipe.predict_proba(samples[feat_cols])[:, 1]

for _, row, p in zip(samples.iterrows(), samples.itertuples(), probas):
    print(f"{row.title:24s} budget=${row.budget:>10,} genre={row.genre:12s} country={row.country} -> "
          f"P(success)={p:.3f} score={round(p*100)} {'SUCCESS' if p >= 0.5 else 'FAILURE'}")

## Next steps

- Deploy the FastAPI service (`ml-service/main.py`) — see `DEPLOYMENT.md`.
- Point the Next.js frontend at `ML_SERVICE_URL` so `/api/predictions` calls `POST /predict`.
- The self-contained TypeScript model in `src/lib/ml/` runs standalone without Python.
